In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.gold
COMMENT 'Capa Gold procesadoss'


In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_departmento")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_departmento (
  departamento_id INT,
  departamento_descripcion STRING
)
""")

In [0]:
import pandas as pd

df = spark.table("workspace.silver.predios_registro").toPandas()

In [0]:

department = (
    df.reindex(
        columns=[
            "departamento_id",
            "departamento_descripcion",
        ]
    )
    .assign(network_type="broadcast")
)

dim = (
    pd.concat([department], ignore_index=True)
    .dropna(subset=["departamento_id"])
    .drop_duplicates(subset=["departamento_id"])
    .sort_values("departamento_descripcion", ignore_index=True)
)

df_spark = spark.createDataFrame(dim)



In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_departmento")